# 5 · Constraints — the integrity features Postgres has always had

Neo4j puts uniqueness, composite and existence constraints behind an enterprise
licence. Postgres has had them forever, and a JSONB property is as constrainable as
a column once the expression is indexed. So they are a first-class feature here
rather than a footnote.

Six declarations, and what each compiles to:

| Declaration | SQL |
| --- | --- |
| `Unique(*keys)` | `CREATE UNIQUE INDEX ON t ((properties->>'k'), ...)` |
| `Unique(..., where=)` | the same, partial — no Neo4j equivalent at any price |
| `Required(*keys)` | `CHECK (properties ?& array['k', ...])` |
| `PropertyType(k, t)` | `CHECK (jsonb_typeof(properties->'k') = t)` |
| `Check(filter)` | `CHECK (<the filter, compiled by the same `resolve()`>)` |
| `Index(*keys)` | `CREATE INDEX ON t ((properties->>'k'), ...)` |

In [1]:
from demo_graph import connect, seed
from hopai import (
    GT, Check, Col, ConstraintViolation, Index, PropertyType, Required, Start, Unique,
)

graph = connect("nb_05_constraints")

## See the SQL before running it

`constraint_ddl()` returns exactly what `define_constraints()` would execute,
without executing it — for review, for a migration file, or for showing an agent
what it is about to change.

In [2]:
declarations = {
    "nodes": [
        Required("type"),                            # the key must be present
        Unique("email"),                             # no two nodes share one
        Unique("tenant", "slug"),                    # composite
        # Partial -- and named, because the auto-derived name comes from the
        # keys alone, so this would otherwise collide with Unique("email") above.
        Unique("email", where={"type": "person"}, name="uq_nodes_person_email"),
        PropertyType("age", "number"),               # not the string "42"
        Check(GT("age", 0), name="age_positive"),    # any filter, as a CHECK
        Index("type"),                               # plain lookup index
    ],
    "edges": [
        Unique(Col("start_id"), Col("end_id"), "kind"),   # one edge of a kind per pair
    ],
}

for statement in graph.constraint_ddl(**declarations):
    print(statement, "\n")

ALTER TABLE "nodes" ADD CONSTRAINT "ck_required_nodes_type" CHECK (graph_id != 'default' OR (properties ?& ARRAY['type'])) 

CREATE UNIQUE INDEX IF NOT EXISTS "uq_nodes_email" ON "nodes" (graph_id, (properties ->> 'email')) 

CREATE UNIQUE INDEX IF NOT EXISTS "uq_nodes_tenant_slug" ON "nodes" (graph_id, (properties ->> 'tenant'), (properties ->> 'slug')) 

CREATE UNIQUE INDEX IF NOT EXISTS "uq_nodes_person_email" ON "nodes" (graph_id, (properties ->> 'email')) WHERE (properties @> CAST('{"type": "person"}' AS JSONB)) 

ALTER TABLE "nodes" ADD CONSTRAINT "ck_number_nodes_age" CHECK (graph_id != 'default' OR jsonb_typeof(properties['age']) = 'number') 

ALTER TABLE "nodes" ADD CONSTRAINT "age_positive" CHECK (graph_id != 'default' OR CAST((properties ->> 'age') AS NUMERIC) > 0) 

CREATE INDEX IF NOT EXISTS "ix_nodes_type" ON "nodes" (graph_id, (properties ->> 'type')) 

CREATE UNIQUE INDEX IF NOT EXISTS "uq_edges_start_id_end_id_kind" ON "edges" (graph_id, start_id, end_id, (properties -

Three details in that output worth pausing on.

**`Col("start_id")` versus `"start_id"`.** A bare string always names a JSONB
property. A real table column has to say so with `Col(...)` — guessing from whether
the name happens to match a column would silently change the meaning of a
constraint on a property called `id`.

**Names are derived from the keys.** `Unique("email")` and
`Unique("email", where=...)` both want to be called `uq_nodes_email`, and the
second `CREATE UNIQUE INDEX IF NOT EXISTS` would then do nothing at all. Pass
`name=` whenever two declarations share their keys — every declaration accepts one.

**`graph_id != 'default' OR ...`** in every CHECK. Constraints are per graph
(notebook 07); an unguarded CHECK would make one graph's rules law for every graph
in the table. The unique indexes get the same treatment by leading with `graph_id`.

## Declaring them

Idempotent, so this belongs next to `create_schema()` in your start-up path.

In [3]:
graph.define_constraints(**declarations)

['ck_required_nodes_type',
 'uq_nodes_email',
 'uq_nodes_tenant_slug',
 'uq_nodes_person_email',
 'ck_number_nodes_age',
 'age_positive',
 'ix_nodes_type',
 'uq_edges_start_id_end_id_kind']

## What a violation looks like

`ConstraintViolation` names the constraint, the table and the offending values —
not a Postgres index name nobody chose. `.constraint`, `.detail` and `.__cause__`
carry the machine-readable parts.

In [4]:
graph.add_nodes([{"type": "person", "name": "Alice", "email": "alice@example.com", "age": 34}])

bad_rows = {
    "duplicate email": {"type": "person", "name": "Imposter", "email": "alice@example.com"},
    "no type at all": {"name": "Anonymous"},
    "age as a string": {"type": "person", "email": "z@example.com", "age": "42"},
    "age of zero": {"type": "person", "email": "zero@example.com", "age": 0},
}

for label, row in bad_rows.items():
    try:
        graph.add_nodes([row])
        print(f"{label:18} accepted (!)")
    except ConstraintViolation as exc:
        print(f"{label:18} {exc.constraint}\n{'':18} {exc}")

duplicate email    uq_nodes_email
                   node rejected by constraint 'uq_nodes_email' -- Key (graph_id, (properties ->> 'email'::text))=(default, alice@example.com) already exists.
no type at all     ck_required_nodes_type
                   node rejected by constraint 'ck_required_nodes_type' -- Failing row contains (3, default, {"name": "Anonymous"}).
age as a string    ck_number_nodes_age
                   node rejected by constraint 'ck_number_nodes_age' -- Failing row contains (4, default, {"age": "42", "type": "person", "email": "z@example.com"}).
age of zero        age_positive
                   node rejected by constraint 'age_positive' -- Failing row contains (5, default, {"age": 0, "type": "person", "email": "zero@example.com"}).


`PropertyType` is worth the line whenever a model writes your data: an LLM emitting
`"42"` where you expected `42` breaks every numeric comparison downstream —
silently, and much later, in a query that just returns fewer rows than it should.

## Two SQL semantics that surprise people once

**1 · A unique index does not constrain rows where the property is missing.**
`properties->>'email'` is NULL for such a row, and NULLs repeat. `Unique("email")`
means "no two nodes share an email", not "everyone has one" — pair it with
`Required("email")` when you mean both. (Neo4j's uniqueness constraint behaves the
same way.)

In [5]:
graph.add_nodes([{"type": "person", "name": "Nemo"},
                 {"type": "person", "name": "Nomen"}])     # two rows, no email, both fine
print("both accepted -- uniqueness says nothing about rows without the key")

both accepted -- uniqueness says nothing about rows without the key


**2 · Postgres evaluates `CHECK` before resolving `ON CONFLICT`.** A merge row must
satisfy every check on its own, even when it is destined to update a row that
already satisfies them. With `Required("type")` declared, a merge that leaves out
`type` is rejected — `ON CONFLICT` resolves uniqueness, not validity.

In [6]:
try:
    graph.merge_nodes([{"email": "alice@example.com", "city": "Berlin"}], on=["email"])
except ConstraintViolation as exc:
    print(f"rejected: {exc}")

# Include the key the CHECK requires and the same merge goes through.
graph.merge_nodes([{"type": "person", "email": "alice@example.com", "city": "Berlin"}],
                  on=["email"])
print(graph.traverse(Start(where={"email": "alice@example.com"})).nodes)

rejected: node rejected by constraint 'ck_required_nodes_type' -- Failing row contains (8, default, {"city": "Berlin", "email": "alice@example.com"}).
[{'id': '1', 'properties': {'age': 34, 'city': 'Berlin', 'name': 'Alice', 'type': 'person', 'email': 'alice@example.com'}}]


Notice what the merge did to the existing row: `city` was added and `name`, `age`
and `type` were left alone. That is `||` over the properties bag — Cypher's
`ON MATCH SET`. Pass `replace=True` to overwrite the whole bag instead.

Merging is idempotent, which is what makes it the right call for an agent that
might retry: run the cell again and nothing changes.

## Partial uniqueness — the one with no Neo4j equivalent

"Email is unique **among people**" is a partial index, and a partial index is just
an index. Companies below may repeat an email that a person already holds:

In [7]:
graph.drop_constraints(nodes=[Unique("email")])    # leaving only uq_nodes_person_email

graph.add_nodes([{"type": "company", "name": "Acme", "email": "alice@example.com"}])
print("company accepted with a person's email -- the surviving index is WHERE type = 'person'")

try:
    graph.add_nodes([{"type": "person", "name": "Clone", "email": "alice@example.com"}])
except ConstraintViolation as exc:
    print(f"second person refused: {exc.constraint}")

company accepted with a person's email -- the surviving index is WHERE type = 'person'
second person refused: uq_nodes_person_email


`drop_constraints()` is the exact inverse of `define_constraints()` — same
declarations in, missing ones ignored.

## Edge constraints work the same way

`Unique(Col("start_id"), Col("end_id"), "kind")` is "at most one edge of a given
kind between the same pair" — the constraint that stops a retried ingestion from
doubling every relationship.

In [8]:
ids = seed(graph.in_graph("edges_demo"))
edges_graph = graph.in_graph("edges_demo")
edges_graph.define_constraints(edges=[Unique(Col("start_id"), Col("end_id"), "kind")])

try:
    edges_graph.add_edges([{"start_id": ids["Alice"], "end_id": ids["Bob"], "kind": "friend"}])
except ConstraintViolation as exc:
    print(f"duplicate edge refused: {exc.constraint}")

duplicate edge refused: uq_edges_start_id_end_id_kind


## Where this leaves you

Every write path is covered, because the database does the checking — `add_nodes`,
`merge_nodes`, a Cypher `CREATE`, or SQL from another service entirely. There is no
"validate before writing" step to forget.

What constraints *cannot* express is anything that has to look at another row —
"a `works_at` edge connects a person to a company" needs the endpoint nodes, and a
CHECK cannot read them. That one is a schema-level rule, which is the next
notebook.

---

Next: [06 · Graph schema](06_graph_schema.ipynb) — declaring the shape of the whole
graph, enforcing it, or inferring it from data you already have.